# Ingest: FDA Drug Adverse Event Reports

Pharma demo data. Pulls drug adverse event reports from the public
openFDA API (no auth, no API key required for this volume) and persists
them as a managed Spark table — one row per (report, reaction) pair,
since a single report can list multiple reactions.

Confirmed live (2026-08-07) field paths:
`results[].safetyreportid`, `.receivedate`, `.serious`,
`.seriousnessdeath`, `.patient.patientonsetage`, `.patient.patientsex`,
`.primarysource.reportercountry`,
`.patient.drug[0].medicinalproduct` (primary suspect drug),
`.patient.reaction[].reactionmeddrapt` (array — exploded below).

Re-runnable: uses `CREATE DATABASE IF NOT EXISTS` and
`mode("overwrite")`, so re-running just pulls a fresh batch of reports.

In [ ]:
import json
import urllib.request

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "pharmacovigilance"
TABLE_NAME = "adverse_events"
API_BASE_URL = "https://api.fda.gov/drug/event.json"
PAGE_SIZE = 100
MAX_PAGES = 10  # 1000 reports -- enough to show real drug/reaction distribution

## Fetch report pages

Offset-paginated via `skip`/`limit`. Each report is exploded into one row
per listed reaction (a report can name several).

In [ ]:
def fetch_adverse_events():
    rows = []
    for page in range(MAX_PAGES):
        skip = page * PAGE_SIZE
        url = f"{API_BASE_URL}?limit={PAGE_SIZE}&skip={skip}"
        with urllib.request.urlopen(url) as resp:
            data = json.load(resp)

        for report in data.get("results", []):
            patient = report.get("patient", {})
            drugs = patient.get("drug", [])
            reactions = patient.get("reaction", [])
            primary_drug = drugs[0].get("medicinalproduct") if drugs else None

            for reaction in reactions:
                rows.append({
                    "safety_report_id": report.get("safetyreportid"),
                    "receive_date": report.get("receivedate"),
                    "serious": report.get("serious"),
                    "seriousness_death": report.get("seriousnessdeath"),
                    "patient_age": patient.get("patientonsetage"),
                    "patient_sex": patient.get("patientsex"),
                    "reporter_country": report.get("primarysource", {}).get("reportercountry"),
                    "drug_name": primary_drug,
                    "reaction_term": reaction.get("reactionmeddrapt"),
                })
    return rows

## Load dataset

In [ ]:
rows = fetch_adverse_events()
df = spark.createDataFrame(rows)

## Create schema/database

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}
COMMENT 'FDA drug adverse event reports used for pharmacovigilance demonstrations'
""")

## Persist as a managed Spark table

In [ ]:
df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TABLE_NAME}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_report_reactions FROM {DATABASE_NAME}.{TABLE_NAME}").show()
spark.sql(f"DESCRIBE {DATABASE_NAME}.{TABLE_NAME}").show(truncate=False)